# Moondream multi-class classification

## Set up and Imports

In [1]:
import os
os.environ['HF_HOME'] = '../cache'

In [2]:
import json

from PIL import Image
from nazi_symbols_classification.training.data_preparation import get_image_paths
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score

## Data Preparation

In [3]:
dir_name = os.path.dirname(os.getcwd())
images = get_image_paths(f"{dir_name}/datasets/nazi-symbols-classification", ("train", "test", "val"))

In [4]:
train_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-classification/train')]
test_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-classification/test')]
valid_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-classification/val')]

In [5]:
y_train = [os.path.basename(os.path.dirname(image)) for image in train_images]
y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

## Model Loading and classification

In [6]:
model = AutoModelForCausalLM.from_pretrained(
"vikhyatk/moondream2",
revision="2025-01-09",
trust_remote_code=True, # Uncomment for GPU acceleration & pip install accelerate # device_map={"": "cuda"}
device_map={"": "cuda"}
)

2025-06-14 18:49:28.751484: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [7]:
def classify_document(doc_path, prompts):
    image = Image.open(doc_path)
    encoded_image = model.encode_image(image)

    result_dict = dict()
    for prompt in prompts:
        answer = model.query(encoded_image, prompt)["answer"]
        print(answer)
        result_dict[prompt] = ("Yes" in answer)

    return result_dict

In [8]:
prompts = {
    "Analyse whether the image contains a black sun symbol, consisting of concentric circles with radiating, rune-like spokes, associated with Nazi occultism?": "black_sun",
    "Analyse whether the image contains the British Union of Fascists logo, a black lightning bolt set within a white circle on a dark background, symbolizing their fascist ideology.": "british_union_of_fascist",
    "Analyse whether the image contains a broken sun cross symbol, featuring a circle divided into four or more segments by straight lines, often associated with white supremacist or neo-Nazi groups.": "broken_sun_cross",
    "Analyse whether the image is one of the historical images of Adolf Hitler addressing crowds, giving speeches, or leading Nazi rallies during the 1930s and 1940s.": "hitler",
    "Analyse whether the image is one of the images of individuals performing the Hitler salute during historical Nazi Germany events, characterized by a raised right arm held at an angle.": "hitler_salute",
    "Analyse whether the image is one of the images of the Judenstern, the yellow Star of David badge used during the Holocaust, often featuring the word 'Jude' in black lettering in the center.": "judenstern",
    "Analyse whether the image is the imagery of neo-Nazi groups featuring hate symbols like swastikas, Black Sun, Siegrune, or Celtic Cross on flags, banners, clothing, or graffiti, often seen at rallies, protests, or in propaganda materials promoting white supremacy and far-right ideology.": "neo-nazi",
    "Analyse whether the image contains a single angular rune shaped like a lightning bolt or elongated 'S,' used in Nazi and neo-Nazi iconography.": "siegrune",
    "Analyse whether the image contains a skull and crossbones insignia, often used by the Nazi SS, with a sinister and militaristic design.": "ss_skull",
    "Analyse whether the image contains an image with the sign of sturmabteilung emblem": "sturmabteilung_emblem",
    "Analyse whether the image contains a black swastika symbol with arms bent at 90 degrees, typically rotated at a 45-degree angle, often shown on a red circular background or a white circle, used during World War II by Nazi Germany.": "swastika",
    "Analyse whether the image contains the Wolfsangel symbol, resembling a hook-like rune, used by Nazi groups and German military units during World War II.": "wolfsangel",
    "Analyse whether the image contains no nazi related content": "non-nazi",
}

In [9]:
%%time
result = dict()

for image_path in test_images:
    classify_result = classify_document(image_path, prompts.keys())
    result[image_path] = classify_result

 Yes
 No
 Yes
 No
 No
 Yes
 No
 Yes
 No
 Yes
 Yes
 Yes
 No
 Yes
 No
 Yes
 No
 Yes
 Yes
 Yes
 Yes
 Yes
 Yes
 No
 Yes
 No
 Yes
 No
 Yes
 Yes
 Yes
 Yes
 Yes
 Yes
 Yes
 Yes
 Yes
 Yes
 No
 No
 No
 Yes
 No
 Yes
 Yes
 No
 Yes
 Yes
 Yes
 No
 Yes
 Yes
 No
 No
 Yes
 No
 Yes
 No
 No
 No
 No
 Yes
 None
 Yes
 No
 No
 Yes
 No
 No
 No
 No
 No
 Single angular rune
 No
 Yes
 No
 Yes
 No
 No
 Yes
 No
 No
 No
 No
 No
 Single angular rune
 No
 Yes
 No
 Yes
 No
 No
 Yes
 No
 No
 No
 No
 No
 Single angular lightning bolt
 No
 Yes
 No
 Yes
 No
 No
 Yes
 No
 Yes
 Yes
 No
 Yes
 Yes
 No
 Yes
 No
 Yes
 No
 No
 Yes
 No
 No
 No
 No
 No
 Single angular rune
 No
 Yes
 No
 Yes
 No
CPU times: user 10.5 s, sys: 181 ms, total: 10.7 s
Wall time: 8.04 s


Store the outputs

In [14]:
to_store = dict(y_true=y_test, y_pred=result)

with open("moondream-output/moondream_result-multi.json", "w") as f:
    json.dump(result, f)

In [13]:
with open("moondream-output/moondream_result-multi.json", "r") as f:
    to_store = json.load(f)

## Evaluation the model based on the classification result from test dataset

In [31]:
outputs = []

for image_path in test_images:
    result_dict = result[image_path]
    labels = []
    for k, v in result_dict.items():
        if not v:
            continue
        labels.append(prompts[k])
    outputs.append(labels)
outputs[:3]

[['black_sun',
  'broken_sun_cross',
  'judenstern',
  'siegrune',
  'sturmabteilung_emblem',
  'swastika',
  'wolfsangel'],
 ['black_sun',
  'broken_sun_cross',
  'hitler_salute',
  'judenstern',
  'neo-nazi',
  'siegrune',
  'ss_skull',
  'sturmabteilung_emblem',
  'wolfsangel'],
 ['black_sun',
  'broken_sun_cross',
  'hitler',
  'hitler_salute',
  'judenstern',
  'neo-nazi',
  'siegrune',
  'ss_skull',
  'sturmabteilung_emblem',
  'swastika',
  'wolfsangel']]

In [15]:
y_test, result = to_store["y_true"], to_store["y_pred"]

In [27]:
replace_dict ={
    "atomwaffen": "neo-nazi", 
    "blood_honor_emblem": "neo-nazi", 
    "celtic_cross": "neo-nazi", 
    "combat_18_emblem": "neo-nazi", 
    "golden_dawn": "neo-nazi", 
    "hammerskins": "neo-nazi", 
    "identitaere_bewegung_emblem": "neo-nazi", 
    "kolovrat": "neo-nazi", 
    "national_rebirth_poland": "neo-nazi", 
    "volksfront_emblem": "neo-nazi", 
    "doppelsiegrune": "siegrune"
}
y_true = [label if label not in replace_dict else replace_dict[label] for label in y_test]

In [28]:
set(y_true)

{'black_sun',
 'british_union_of_fascist',
 'broken_sun_cross',
 'happy_merchant',
 'hitler',
 'hitler_salute',
 'judenstern',
 'neo-nazi',
 'siegrune',
 'ss_skull',
 'sturmabteilung_emblem',
 'swastika',
 'wolfsangel'}

In [32]:
y_pred = []
for i, label in enumerate(y_test):
    if label in outputs[i]:
        y_pred.append(label)
    elif not outputs[i]:
        y_pred.append("non-nazi")
    else:
        y_pred.append(outputs[i][0])
y_pred[:3]

['black_sun', 'hitler_salute', 'hitler_salute']

Print Classification report and calculate metrics.

In [35]:
print(classification_report(y_true, y_pred, digits=3))

                          precision    recall  f1-score   support

               black_sun      0.535     0.855     0.658       124
british_union_of_fascist      0.220     0.917     0.355        12
        broken_sun_cross      0.077     0.688     0.139        16
          happy_merchant      0.000     0.000     0.000        33
                  hitler      0.933     0.227     0.365       185
           hitler_salute      0.046     0.800     0.087         5
              judenstern      0.022     1.000     0.043         3
                neo-nazi      0.072     0.051     0.059       158
                non-nazi      0.000     0.000     0.000         0
                siegrune      0.904     0.287     0.436       261
                ss_skull      0.967     0.791     0.870       220
   sturmabteilung_emblem      0.050     1.000     0.095         8
                swastika      1.000     0.799     0.888       886
              wolfsangel      0.943     1.000     0.971        33

        

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.


In [36]:
accuracy_score(y_true, y_pred)

0.6085390946502057